# 23. Ensemble Learning: XGBoost (Extreme Gradient Boosting)

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: High  
**Use Case**: Optimized gradient boosting with advanced regularization, parallel processing, and state-of-the-art performance

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the theoretical foundations of gradient boosting and XGBoost's extensions
- Comprehend second-order (Newton) boosting and how XGBoost uses Hessians
- Understand the regularized objective function and tree construction algorithm
- Master XGBoost's API: XGBClassifier, XGBRegressor, DMatrix, and low-level xgb.train()
- Implement XGBoost for classification and regression with best practices
- Use early stopping, cross-validation, and hyperparameter tuning effectively
- Handle imbalanced data using scale_pos_weight and other techniques
- Compare XGBoost with LightGBM and CatBoost to choose the right tool
- Evaluate models comprehensively with multiple metrics and diagnostic tools
- Interpret models using feature importance, permutation importance, and SHAP values
- Deploy XGBoost models in production environments
- Create custom objective functions and evaluation metrics
- Optimize XGBoost for speed and memory efficiency
- Apply XGBoost to real-world problems with confidence

## Historical Context

XGBoost was developed by Tianqi Chen and Carlos Guestrin in 2016:
- **Chen, T. & Guestrin, C. (2016)**: "XGBoost: A Scalable Tree Boosting System"
- Won numerous Kaggle competitions and became the go-to algorithm for structured data
- Optimized for speed, memory efficiency, and performance
- Introduced key innovations: regularization, second-order gradients, sparsity awareness, parallelization
- Foundation for many modern gradient boosting frameworks

**Key Papers/References:**
- Chen, T. & Guestrin, C. (2016). "XGBoost: A Scalable Tree Boosting System" (KDD 2016)
- XGBoost Documentation: https://xgboost.readthedocs.io/
- Various optimization and implementation guides from the XGBoost community

## When to Use XGBoost

XGBoost is appropriate when:
- You need the highest possible accuracy on structured/tabular data
- Working with large datasets (millions of rows, thousands of features)
- You have computational resources (multi-core CPU, optionally GPU)
- Both classification and regression tasks
- Competition-level performance is required
- You need robust handling of missing values and sparse data
- You want a well-tested, reliable solution with extensive community support
- You need model interpretability through feature importance and SHAP values

**Consider alternatives when:**
- You have primarily categorical features (consider CatBoost)
- Training speed is critical on very large data (consider LightGBM)
- You need real-time predictions with minimal latency
- Working with unstructured data (images, text) - use neural networks
- You prefer minimal hyperparameter tuning (CatBoost has better defaults)

## Theory & Mechanics

### Mathematical Foundation

XGBoost extends gradient boosting with regularization, second-order information, and numerous optimizations. Understanding the mathematical foundation is crucial for effective use.

#### Gradient Boosting Basics

**Boosting Overview:**
Boosting is an ensemble technique where multiple weak learners (typically decision trees) are trained sequentially. Each new model focuses on correcting the errors of the previous ensemble.

**Gradient Boosting Mechanics:**
At iteration $t$, suppose the current model (ensemble of $t-1$ trees) produces predictions $\hat{y}_i^{(t-1)}$ for each training instance $i$. We want to add a new tree $f_t(x)$ to minimize a loss function $L$.

Using a first-order approximation, the negative gradient of the loss with respect to the predictions gives the direction of steepest decrease. For many losses, this negative gradient is just the residual (true value minus prediction):

$$g_i = \frac{\partial L(y_i, \hat{y}_i^{(t-1)})}{\partial \hat{y}_i^{(t-1)}}$$

Gradient boosting fits $f_t(x)$ to these negative gradients (or residuals), meaning each new tree predicts the correction needed for the current model.

#### XGBoost's Second-Order (Newton) Boosting

XGBoost extends this by also using **second-order gradients (Hessians)** - a technique called Newton boosting. This provides more accurate estimates of the optimal update and leads to faster convergence.

The Hessian (second derivative) is:
$$h_i = \frac{\partial^2 L(y_i, \hat{y}_i^{(t-1)})}{\partial (\hat{y}_i^{(t-1)})^2}$$

XGBoost uses both $g_i$ and $h_i$ when fitting new trees, incorporating a second-order Taylor approximation of the loss function for each split. This makes the algorithm more efficient and often more accurate than first-order-only methods.

#### Regularized Objective Function

A core difference in XGBoost's formulation is the inclusion of a **regularized objective function**. In standard gradient boosting, we minimize the loss $L$. XGBoost adds a regularization term $\Omega$ to penalize model complexity and avoid overfitting.

**XGBoost Objective at iteration $t$:**
$$Obj^{(t)} = \sum_{i=1}^{n} L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

Where:
- $L(y_i, \hat{y}_i)$: Loss function (e.g., squared error for regression, log-loss for classification)
- $\Omega(f_t)$: Regularization term for tree $f_t$

**Regularization Term:**
$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2 + \alpha \sum_{j=1}^{T} |w_j|$$

Where:
- $T$: Number of leaves in the tree
- $w_j$: Weight (score) on leaf $j$
- $\gamma$: Minimum loss reduction required for a split (complexity cost)
- $\lambda$: L2 regularization on leaf weights
- $\alpha$: L1 regularization on leaf weights

By penalizing the number of leaves and the magnitude of weights, the algorithm favors simpler trees that generalize better. This is key to XGBoost's ability to combat overfitting while maintaining high flexibility.

#### Tree Construction Algorithm

XGBoost builds decision trees by choosing the best split at each node based on an impurity or gain measure. However, XGBoost's splitter uses gradient and Hessian statistics to compute the gain more precisely.

**Split Gain Calculation:**
For a potential split, XGBoost calculates the reduction in loss (objective) that would result, including the regularization term:

$$Gain = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma$$

Where:
- $G_L, G_R$: Sum of gradients in left and right child nodes
- $H_L, H_R$: Sum of Hessians in left and right child nodes
- $\gamma$: Minimum gain required (if gain < $\gamma$, split is not made)

The algorithm enumerates over possible split points and picks the split with the highest positive gain. If no split can reduce the loss by at least $\gamma$, tree construction stops at that node.

**Tree Growth Strategy:**
- XGBoost uses **max_depth** to limit tree growth (unlike some frameworks that use number of leaves)
- Trees are grown **depth-first** up to max_depth
- Any branch with insufficient gain can be **pruned prematurely**
- This depth-first "prune" strategy makes training efficient by allocating resources to promising splits only

#### Missing Value Handling

XGBoost is **sparsity-aware**: when a feature value is missing (NaN) or a sparse matrix has a zero, XGBoost can assign a default direction to treat that missing value in tree splitting.

During training, XGBoost finds the optimal default direction (left or right branch) for missing values based on which choice yields less loss. This means you don't need to impute missing data - XGBoost will learn how to route missing values in a way that minimizes the objective.

This feature is extremely useful for real-world datasets with missing entries.

### How It Works: Step-by-Step

1. **Initialize**: Start with base prediction (e.g., mean for regression, log-odds for classification)
2. **For each boosting round**:
   - Calculate first-order gradients ($g_i$) and second-order gradients ($h_i$) for all training instances
   - Build a new tree $f_t$:
     - For each feature, enumerate possible split points
     - Calculate gain for each split using gradient and Hessian statistics
     - Choose split with highest gain (if gain > $\gamma$)
     - Continue until max_depth reached or no beneficial splits
     - Prune branches with negative gain
   - Add tree to ensemble: $\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)$
     - $\eta$ is the learning rate (shrinkage)
3. **Regularize**: L1/L2 penalties are applied during tree construction (in gain calculation)
4. **Predict**: Sum predictions from all trees: $\hat{y}_i = \sum_{k=1}^{K} f_k(x_i)$

### Key Hyperparameters

**Core Parameters:**
- **n_estimators** (num_boost_round): Number of boosting rounds (trees)
- **learning_rate** (eta): Step size shrinkage (0-1). Lower values require more trees but can improve accuracy
- **max_depth**: Maximum tree depth. Controls model complexity - deeper trees can fit more patterns but may overfit
- **min_child_weight**: Minimum sum of instance weights (Hessian) needed in a child node. Prevents creating nodes with too few samples
- **gamma** (min_split_loss): Minimum loss reduction required to make a split. Higher values = more conservative (shallower trees)

**Sampling Parameters:**
- **subsample**: Fraction of training instances to sample for each tree (0-1). < 1.0 adds randomness (stochastic gradient boosting)
- **colsample_bytree**: Fraction of features to sample for each tree
- **colsample_bylevel**: Fraction of features to sample at each tree level
- **colsample_bynode**: Fraction of features to sample at each node

**Regularization Parameters:**
- **reg_alpha** (alpha): L1 regularization on leaf weights. Can drive some weights to exactly 0 (feature selection)
- **reg_lambda** (lambda): L2 regularization on leaf weights. Default is 1.0, higher values = more conservative

**Other Important Parameters:**
- **scale_pos_weight**: For binary classification with imbalanced data. Scales gradient for positive class
- **tree_method**: Algorithm for tree construction ('exact', 'approx', 'hist', 'gpu_hist')
- **objective**: Loss function to optimize (e.g., 'binary:logistic', 'reg:squarederror')
- **eval_metric**: Evaluation metric to monitor (e.g., 'logloss', 'auc', 'rmse')

### Advantages

- **Very high accuracy**: Often achieves state-of-the-art performance on structured data
- **Fast training**: Parallelization and optimized algorithms make it efficient
- **Handles missing values**: Automatically learns optimal treatment of missing data
- **Built-in regularization**: L1/L2 penalties help prevent overfitting
- **Early stopping support**: Prevents overfitting by stopping when validation performance plateaus
- **Feature importance**: Provides multiple types of feature importance scores
- **Sparsity awareness**: Efficient with sparse data (one-hot encoded categories, text features)
- **Flexible**: Supports custom objectives and evaluation metrics
- **Well-tested**: Extensive use in competitions and production systems

### Limitations

- **Requires hyperparameter tuning**: Many parameters to tune for optimal performance
- **Can overfit**: If not tuned properly, especially with deep trees and many estimators
- **Memory intensive**: Large models with many trees can consume significant memory
- **Less interpretable**: Complex ensemble is harder to interpret than single trees
- **Computational cost**: Training can be slow on very large datasets (though optimized)
- **Sensitive to outliers**: Tree-based methods can be affected by extreme values
- **Not ideal for unstructured data**: Designed for tabular/structured data, not images/text directly


## Key Features & Innovations

XGBoost didn't become popular just by chance – it introduced several important innovations and engineering optimizations on top of basic gradient boosting:

### 1. Regularization

XGBoost includes built-in **L1 (Lasso)** and **L2 (Ridge)** regularization on tree parameters (leaf weights) and an additional penalty for the number of leaves. This helps prevent overfitting, especially in deep trees.

- **reg_alpha** (L1): Can drive some feature weights to exactly 0, effectively doing feature selection
- **reg_lambda** (L2): Encourages smaller weights, making the model more conservative
- **gamma**: Penalty for each additional leaf, encouraging simpler trees

Traditional gradient boosting (as in sklearn's GradientBoostingClassifier) did not explicitly include L1/L2 penalties on tree weights. XGBoost's inclusion of these makes the model more robust and less prone to overfitting.

### 2. Second-Order (Newton) Boosting

XGBoost's tree construction uses not only first-order gradients but also **second-order derivatives (Hessians)** to guide split finding. The use of second-order information gives a more accurate estimation of the optimal split gain and can improve convergence speed.

In effect, XGBoost performs a Newton-like update within each boosting iteration rather than simple gradient descent. This means:
- Faster convergence (fewer trees needed)
- More accurate split decisions
- Better handling of complex loss landscapes

### 3. Tree Pruning

XGBoost employs a **max_depth** limit and a **prune after split** strategy. It explores splits up to the specified max_depth, but after growing the tree, it will prune backward and remove splits that failed to produce a positive gain.

The **gamma** parameter controls the minimum gain for a split: higher gamma values result in more aggressive pruning (requiring larger reduction in loss to keep a split). This ensures that each tree added actually improves the model's objective.

### 4. Handling Missing Values

XGBoost has built-in capability to handle missing data internally. When XGBoost encounters a missing value for a feature, it doesn't discard the instance; instead, it tries sending the instance down either left or right branch and learns which path minimizes the loss.

Essentially, it assigns a **default direction** for missing values at each split by optimizing the training loss. This means you can feed XGBoost raw data with NaNs and it will handle them automatically. This is a big advantage in data preprocessing effort.

### 5. Sparsity-Aware Learning

Many datasets (especially those with one-hot encoded categorical variables or text features) are sparse – meaning most feature values are 0 for a given instance. XGBoost implements a "sparsity-aware" tree building algorithm that efficiently skips over missing/zero entries in data.

Internally, the **DMatrix** (XGBoost's optimized data structure) stores data in a compressed sparse column format and pre-sorts features, so that XGBoost can quickly iterate only over the non-zero entries of a feature when calculating split gains. This significantly speeds up computation on sparse data and saves memory.

### 6. Parallelization

One of XGBoost's slogans is that it's "fast" – and a major reason is its ability to utilize hardware resources. XGBoost is implemented in C++ with optimized code. By default, it will use all available CPU cores to parallelize certain operations.

Specifically, while growing a tree, finding the best split for each node can be parallelized by dividing the work across features (each thread can compute the best split for a subset of features). Once the best split overall is found, the tree is split and the process repeats. This is multi-threading at the algorithmic level.

XGBoost also supports distributed training across multiple machines (using frameworks like Rabit or integrations with Spark and Dask), though that's beyond our scope here.

### 7. Efficient Memory Usage

XGBoost uses a number of tricks to reduce memory overhead:
- **DMatrix structure**: Memory-efficient data storage
- **Memory pooling**: Avoids unnecessary data copies
- **Out-of-core computation**: For very large datasets that don't fit in memory, XGBoost can perform block processing that handles data from disk in batches
- **Cache-aware block structure**: Aligns data with CPU cache boundaries for faster access

These low-level optimizations allow XGBoost to scale to big data settings where other libraries might run out of memory or thrash the disk.

### 8. Approximate Learning (Weighted Quantile Sketch)

For extremely large datasets, sorting all feature values to find split points can be slow. XGBoost introduces an approximate splitting method using a **weighted quantile sketch algorithm**.

In essence, it can efficiently propose candidate splits without sorting every value by building quantile summaries of the data. This comes into play when you use `tree_method='approx'` or `tree_method='hist'` (histogram-based algorithm, similar to LightGBM's approach).

The weighted quantile sketch ensures that even with approximation, the chosen splits are nearly as good as exact greedy splits, but it runs much faster on huge data.

### 9. Early Stopping

Although not unique to XGBoost, it's worth noting that XGBoost's training API supports early stopping out-of-the-box. If you provide a validation set and a patience (`early_stopping_rounds`), XGBoost will monitor the validation metric and stop training when it doesn't improve for a given number of rounds.

This helps to avoid overfitting by halting at the optimal point. We'll see examples of this in code later.

### Summary

XGBoost's key innovations include "a novel tree learning algorithm for handling sparse data, a more regularized model formalization to control over-fitting, and efficient use of computational resources", along with features like built-in cross-validation, missing value handling, and parallelization. It's these enhancements that make XGBoost extremely powerful in practice, often delivering top performance in machine learning competitions and real-world applications.


## XGBoost API: XGBClassifier, XGBRegressor, DMatrix, and Parameters

XGBoost provides a rich Python API that is well-integrated with the scikit-learn ecosystem. Understanding the different APIs and when to use them is important for effective XGBoost usage.


### Scikit-Learn API: XGBClassifier and XGBRegressor

The two most commonly used high-level classes are:

- **XGBClassifier**: for classification tasks (binary or multiclass)
- **XGBRegressor**: for regression tasks

These classes are scikit-learn estimators, meaning they implement the familiar `.fit()` / `.predict()` interface and can be used in sklearn.model_selection routines (grid search, pipelines, etc.). Under the hood, they are built on top of XGBoost's core training functions but simplify many details for you.


### DMatrix: XGBoost's Optimized Data Structure

**DMatrix** is a fundamental data structure in XGBoost that holds data in an optimized way. It is not mandatory to use DMatrix with the scikit-learn API (as it will be handled internally), but it's good to understand for advanced usage.

A DMatrix can be constructed from NumPy arrays, Pandas DataFrames, or SciPy sparse matrices. DMatrix is optimized for both memory efficiency and training speed. It will automatically handle tasks like missing value masking, pre-sorting of features, and compressing sparse data.

Using a DMatrix can significantly reduce memory usage and training time, especially with sparse data. In fact, even when you call `XGBClassifier.fit()`, the data is internally converted to a DMatrix (so you're benefiting from it implicitly).

The DMatrix also allows setting additional per-example weights, base margin (initial prediction), and grouping information for ranking tasks.


### Low-Level API: xgb.train()

In addition to the sklearn-style classes, XGBoost has a lower-level API where you explicitly use `xgb.DMatrix` and `xgb.train()` or `xgb.cv()`. This gives you more control and is useful for custom objectives or when needing more fine-grained control over training.

The low-level API is particularly useful for:
- Custom objective functions
- Custom evaluation metrics
- More control over training callbacks
- Direct access to Booster object


## ROC and Precision-Recall Curves

Let's visualize the model's performance using ROC and Precision-Recall curves.


In [ ]:
# ROC Curve
roc_auc, fpr, tpr = plot_roc_curve(y_test.values, y_pred_proba, 
                                    title="XGBoost ROC Curve")

# Precision-Recall Curve
avg_precision, precision, recall = plot_precision_recall_curve(
    y_test.values, y_pred_proba, 
    title="XGBoost Precision-Recall Curve"
)

print(f"ROC-AUC Score: {roc_auc:.3f}")
print(f"Average Precision: {avg_precision:.3f}")

# Confusion Matrix Visualization
plot_confusion_matrix(y_test.values, y_pred, 
                     class_names=cancer.target_names,
                     title="XGBoost Confusion Matrix")


## Cross-Validation

Let's perform cross-validation to get a more robust estimate of model performance.


In [ ]:
# 5-fold Cross-Validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")
print(f"  Individual fold scores: {cv_scores}")

# Check stability
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"\nCV Stability Check:")
print(f"  Coefficient of Variation: {stability['cv_coefficient']:.3f}")
print(f"  Is Stable: {stability['is_stable']}")

# Visualize CV scores
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_scores, alpha=0.7, color='skyblue')
plt.axhline(y=cv_mean, color='r', linestyle='--', 
           label=f'Mean: {cv_mean:.3f}')
plt.fill_between(range(1, 6), cv_mean - cv_std, cv_mean + cv_std, 
                 alpha=0.2, color='red', label=f'±1 std: {cv_std:.3f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## Using DMatrix (Low-Level API)

Let's demonstrate the low-level API using DMatrix for more control.


In [ ]:
# Create DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# Define parameters
params = {
    'objective': 'binary:logistic',
    'max_depth': 3,
    'learning_rate': 0.1,
    'eval_metric': 'logloss',
    'random_state': 42
}

# Create watchlist for monitoring
watchlist = [(dtrain, 'train'), (dval, 'eval')]

# Train using low-level API
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    early_stopping_rounds=10,
    verbose_eval=False
)

# Make predictions
y_pred_dmatrix = bst.predict(dtest)
y_pred_dmatrix_binary = (y_pred_dmatrix > 0.5).astype(int)
accuracy_dmatrix = accuracy_score(y_test, y_pred_dmatrix_binary)

print("Low-Level API Results:")
print(f"  Test Accuracy: {accuracy_dmatrix:.3f}")
print(f"  Best iteration: {bst.best_iteration}")
print(f"  Best score: {bst.best_score:.4f}")

# Compare with sklearn API
print(f"\nComparison:")
print(f"  Sklearn API accuracy: {accuracy:.3f}")
print(f"  Low-level API accuracy: {accuracy_dmatrix:.3f}")
print(f"  Difference: {abs(accuracy - accuracy_dmatrix):.4f}")


## Best Practices: Hyperparameter Tuning, Early Stopping, Cross-Validation, and Imbalanced Data

Training an XGBoost model is relatively straightforward, but getting the most out of it often requires careful tuning and best practices.


### Hyperparameter Tuning Strategy

XGBoost has many hyperparameters, and tuning them can significantly impact performance. Here's a systematic approach:


In [ ]:
# Strategy 1: Start with defaults, then tune high-impact parameters
# Trade-off: learning_rate and n_estimators
# Lower learning_rate typically means you need more trees

print("Hyperparameter Tuning Strategy:")
print("1. Start with learning_rate=0.1 and large n_estimators (500-1000)")
print("2. Use early_stopping_rounds to find optimal number of trees")
print("3. Tune max_depth, min_child_weight, gamma for complexity control")
print("4. Tune subsample and colsample_bytree for regularization")
print("5. Finally tune reg_alpha and reg_lambda if overfitting persists")

# Example: Learning rate vs number of trees
learning_rates = [0.01, 0.05, 0.1, 0.2]
results_lr = []

for lr in learning_rates:
    xgb_test = XGBClassifier(
        n_estimators=200,
        learning_rate=lr,
        max_depth=3,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_test.fit(X_train, y_train)
    pred = xgb_test.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results_lr.append({'lr': lr, 'accuracy': acc})
    print(f"  Learning rate {lr}: Accuracy = {acc:.3f}")

# Visualize
lr_df = pd.DataFrame(results_lr)
plt.figure(figsize=(10, 5))
plt.plot(lr_df['lr'], lr_df['accuracy'], 'o-', markersize=8)
plt.xlabel('Learning Rate')
plt.ylabel('Test Accuracy')
plt.title('Effect of Learning Rate on Performance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: Lower learning rates often achieve better accuracy but require more trees.")
print("Use early stopping to find the optimal number of trees for each learning rate.")


### Handling Imbalanced Data

Class imbalance can severely affect model training. XGBoost offers several tools for this:


In [ ]:
# Create imbalanced dataset for demonstration
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=10,
    n_clusters_per_class=1,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    random_state=42
)

X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

print("Imbalanced Dataset:")
print(f"  Class distribution (train): {pd.Series(y_imb_train).value_counts().to_dict()}")
print(f"  Class distribution (test): {pd.Series(y_imb_test).value_counts().to_dict()}")

# Without scale_pos_weight
model_no_weight = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
model_no_weight.fit(X_imb_train, y_imb_train)
y_pred_no_weight = model_no_weight.predict(X_imb_test)
acc_no_weight = accuracy_score(y_imb_test, y_pred_no_weight)

# Calculate scale_pos_weight (negative_count / positive_count)
negative_count = (y_imb_train == 0).sum()
positive_count = (y_imb_train == 1).sum()
scale_pos_weight = negative_count / positive_count

print(f"\nCalculated scale_pos_weight: {scale_pos_weight:.2f}")

# With scale_pos_weight
model_weighted = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
model_weighted.fit(X_imb_train, y_imb_train)
y_pred_weighted = model_weighted.predict(X_imb_test)
acc_weighted = accuracy_score(y_imb_test, y_pred_weighted)

# Compare metrics
metrics_no_weight = calculate_classification_metrics(y_imb_test, y_pred_no_weight)
metrics_weighted = calculate_classification_metrics(y_imb_test, y_pred_weighted)

print("\nComparison: Without vs With scale_pos_weight")
print(f"  Accuracy:")
print(f"    Without weight: {acc_no_weight:.3f}")
print(f"    With weight: {acc_weighted:.3f}")
print(f"\n  Precision:")
print(f"    Without weight: {metrics_no_weight['precision']:.3f}")
print(f"    With weight: {metrics_weighted['precision']:.3f}")
print(f"\n  Recall:")
print(f"    Without weight: {metrics_no_weight['recall']:.3f}")
print(f"    With weight: {metrics_weighted['recall']:.3f}")
print(f"\n  F1 Score:")
print(f"    Without weight: {metrics_no_weight['f1_score']:.3f}")
print(f"    With weight: {metrics_weighted['f1_score']:.3f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrices
cm_no_weight = confusion_matrix(y_imb_test, y_pred_no_weight)
cm_weighted = confusion_matrix(y_imb_test, y_pred_weighted)

axes[0].imshow(cm_no_weight, interpolation='nearest', cmap='Blues')
axes[0].set_title('Without scale_pos_weight')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_no_weight[i, j]), ha='center', va='center')

axes[1].imshow(cm_weighted, interpolation='nearest', cmap='Blues')
axes[1].set_title('With scale_pos_weight')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm_weighted[i, j]), ha='center', va='center')

plt.tight_layout()
plt.show()

print("\nNote: scale_pos_weight improves recall on minority class (fewer false negatives)")
print("but may slightly reduce precision (more false positives). This is often desirable")
print("for imbalanced problems where missing the minority class is costly.")


### Systematic Hyperparameter Tuning with GridSearchCV

For production models, systematic hyperparameter optimization is essential:


In [ ]:
# Hyperparameter tuning with GridSearchCV
# Note: This is a reduced grid for demonstration - in practice, use RandomizedSearchCV
# or Optuna for larger parameter spaces

param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1.0, 1.5]
}

print("Grid Search Parameters:")
print(f"  Total combinations: {np.prod([len(v) for v in param_grid.values()])}")
print("  This will take some time...")

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=1,  # Set to 1 to avoid overloading CPU (XGBoost uses all cores internally)
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Accuracy: {grid_search.best_score_:.3f}")

# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"Test Accuracy (best model): {acc_best:.3f}")
print(f"Improvement over default: {acc_best - accuracy:.3f}")

# Compare parameter importance
results_df = pd.DataFrame(grid_search.cv_results_)
print(f"\nTop 3 Parameter Combinations:")
top_3 = results_df.nlargest(3, 'mean_test_score')[['params', 'mean_test_score', 'std_test_score']]
for idx, row in top_3.iterrows():
    print(f"  Score: {row['mean_test_score']:.3f} (+/- {row['std_test_score']:.3f})")
    print(f"    Params: {row['params']}")


## XGBoost vs LightGBM vs CatBoost: When to Use Which?

In the realm of gradient boosting machines (GBMs), XGBoost isn't the only player. LightGBM and CatBoost are two other popular frameworks. Each has its own strengths and ideal use-cases.


### Comparison Overview

Let's compare the three frameworks on key dimensions:


In [ ]:
# Comparison table
comparison_data = {
    'Framework': ['XGBoost', 'LightGBM', 'CatBoost'],
    'Speed': ['Fast', 'Fastest', 'Slow (CPU), Fast (GPU)'],
    'Memory': ['Moderate', 'Low', 'Moderate'],
    'Categorical Features': ['Requires encoding', 'Native support', 'Excellent native support'],
    'Ease of Tuning': ['Moderate', 'Moderate', 'Easy (good defaults)'],
    'Accuracy': ['Excellent', 'Excellent', 'Excellent'],
    'Best For': [
        'Well-tested, reliable solution',
        'Very large datasets, speed critical',
        'Categorical-heavy data, minimal tuning'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("Framework Comparison:")
print(comparison_df.to_string(index=False))

# Try to import LightGBM and CatBoost for actual comparison
# (They may not be installed, so we'll handle gracefully)
try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("\nNote: LightGBM not available. Install with: pip install lightgbm")

try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("Note: CatBoost not available. Install with: pip install catboost")

# If available, do a quick comparison
if LIGHTGBM_AVAILABLE:
    print("\n" + "="*60)
    print("Quick Speed/Accuracy Comparison (if libraries available):")
    print("="*60)
    
    # XGBoost
    import time
    start = time.time()
    xgb_model = XGBClassifier(n_estimators=50, random_state=42, eval_metric='logloss', use_label_encoder=False)
    xgb_model.fit(X_train, y_train)
    xgb_time = time.time() - start
    xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
    
    # LightGBM
    start = time.time()
    lgb_model = lgb.LGBMClassifier(n_estimators=50, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)
    lgb_time = time.time() - start
    lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test))
    
    print(f"\nXGBoost:")
    print(f"  Training time: {xgb_time:.3f}s")
    print(f"  Test accuracy: {xgb_acc:.3f}")
    
    print(f"\nLightGBM:")
    print(f"  Training time: {lgb_time:.3f}s")
    print(f"  Test accuracy: {lgb_acc:.3f}")
    print(f"  Speedup: {xgb_time/lgb_time:.2f}x")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].bar(['XGBoost', 'LightGBM'], [xgb_time, lgb_time], alpha=0.7, color=['blue', 'green'])
    axes[0].set_ylabel('Training Time (seconds)')
    axes[0].set_title('Training Speed Comparison')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(['XGBoost', 'LightGBM'], [xgb_acc, lgb_acc], alpha=0.7, color=['blue', 'green'])
    axes[1].set_ylabel('Test Accuracy')
    axes[1].set_title('Accuracy Comparison')
    axes[1].set_ylim([0.9, 1.0])
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n" + "="*60)
    print("Framework Selection Guide:")
    print("="*60)
    print("\nUse XGBoost if:")
    print("  - You want a well-tested, reliable solution")
    print("  - You need certain advanced features (wide selection of loss functions)")
    print("  - You need to deploy to Java/C++ environments easily")
    print("  - You're following Kaggle solutions (many use XGBoost)")
    
    print("\nUse LightGBM if:")
    print("  - Training speed or memory is a bottleneck")
    print("  - You have extremely large datasets (millions of rows)")
    print("  - You need quick prototyping with limited computation time")
    
    print("\nUse CatBoost if:")
    print("  - You have lots of categorical features")
    print("  - You want to avoid complex encoding pipelines")
    print("  - You observe overfitting with other GBMs")
    print("  - You want something that works well with minimal tuning")


## Model Evaluation & Diagnostics

Evaluating an XGBoost model requires appropriate metrics and diagnostic tools. Let's explore comprehensive evaluation techniques.


### Classification Metrics Deep Dive


In [ ]:
# Comprehensive classification evaluation
y_pred_proba_full = model.predict_proba(X_test)

# ROC AUC
roc_auc = roc_auc_score(y_test, y_pred_proba_full[:, 1])
print("Classification Metrics Summary:")
print(f"  Accuracy: {accuracy:.3f}")
print(f"  ROC-AUC: {roc_auc:.3f}")
print(f"  Precision: {metrics['precision']:.3f}")
print(f"  Recall: {metrics['recall']:.3f}")
print(f"  F1 Score: {metrics['f1_score']:.3f}")

# Average Precision (PR AUC)
avg_precision = average_precision_score(y_test, y_pred_proba_full[:, 1])
print(f"  Average Precision (PR-AUC): {avg_precision:.3f}")

# Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Threshold analysis for binary classification
thresholds = np.arange(0.1, 1.0, 0.1)
threshold_results = []

for threshold in thresholds:
    y_pred_thresh = (y_pred_proba_full[:, 1] > threshold).astype(int)
    acc = accuracy_score(y_test, y_pred_thresh)
    prec = precision_score(y_test, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    threshold_results.append({
        'threshold': threshold,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    })

threshold_df = pd.DataFrame(threshold_results)

# Plot threshold analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(threshold_df['threshold'], threshold_df['precision'], 'o-', label='Precision', markersize=6)
axes[0].plot(threshold_df['threshold'], threshold_df['recall'], 's-', label='Recall', markersize=6)
axes[0].plot(threshold_df['threshold'], threshold_df['f1'], '^-', label='F1', markersize=6)
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision-Recall-F1 vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(threshold_df['threshold'], threshold_df['accuracy'], 'o-', color='green', markersize=6)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs Threshold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: For imbalanced data, you might want to use a threshold other than 0.5")
print("to optimize for precision or recall based on your use case.")


### Training vs Validation Curves

Monitoring training progress helps detect overfitting:


In [ ]:
# Train a model with many trees to observe overfitting
model_overfit = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,  # Deeper trees
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

model_overfit.fit(
    X_train_split, y_train_split,
    eval_set=[(X_train_split, y_train_split), (X_val, y_val)],
    verbose=False
)

# Get evaluation history
evals_result_overfit = model_overfit.evals_result()
train_loss = evals_result_overfit['validation_0']['logloss']
val_loss = evals_result_overfit['validation_1']['logloss']

# Plot training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Training Loss', alpha=0.7, linewidth=2)
plt.plot(val_loss, label='Validation Loss', alpha=0.7, linewidth=2)
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Calculate gap (overfitting indicator)
gap = np.array(train_loss) - np.array(val_loss)
plt.subplot(1, 2, 2)
plt.plot(gap, label='Train - Val Gap', color='red', linewidth=2)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Boosting Round')
plt.ylabel('Loss Gap')
plt.title('Overfitting Indicator (Positive = Overfitting)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal stopping point
optimal_round = np.argmin(val_loss)
print(f"Optimal stopping point: Round {optimal_round}")
print(f"  Training loss at optimal: {train_loss[optimal_round]:.4f}")
print(f"  Validation loss at optimal: {val_loss[optimal_round]:.4f}")
print(f"  Gap at optimal: {gap[optimal_round]:.4f}")

if gap[-1] > 0.1:
    print(f"\n⚠ Warning: Significant overfitting detected (gap = {gap[-1]:.4f})")
    print("  Consider: reducing max_depth, increasing regularization, or using early stopping")
else:
    print(f"\n✓ Model shows good generalization (gap = {gap[-1]:.4f})")


## Model Interpretability: Feature Importance, Permutation Importance, and SHAP Values

Ensemble tree models like XGBoost are often considered "black boxes," but there are several techniques to interpret and explain their predictions.


### Permutation Importance

Permutation importance is a model-agnostic method that measures feature importance by shuffling features and observing the drop in performance.


In [ ]:
# Permutation Importance
print("Calculating Permutation Importance (this may take a moment)...")
perm_importance = permutation_importance(
    model, X_test, y_test,
    scoring='accuracy',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

# Get feature names
feature_names = X.columns.tolist()
perm_means = perm_importance.importances_mean
perm_std = perm_importance.importances_std

# Create DataFrame
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_means,
    'importance_std': perm_std
}).sort_values('importance_mean', ascending=False)

print("\nTop 10 Features by Permutation Importance:")
print(perm_df.head(10).to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Permutation importance
top_perm = perm_df.head(10)
axes[0].barh(range(len(top_perm)), top_perm['importance_mean'], xerr=top_perm['importance_std'], alpha=0.7)
axes[0].set_yticks(range(len(top_perm)))
axes[0].set_yticklabels(top_perm['feature'])
axes[0].set_xlabel('Importance (Accuracy Drop)')
axes[0].set_title('Top 10 Features: Permutation Importance')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Compare with XGBoost's built-in importance
top_xgb = feature_importance.head(10)
axes[1].barh(range(len(top_xgb)), top_xgb['importance'], alpha=0.7, color='green')
axes[1].set_yticks(range(len(top_xgb)))
axes[1].set_yticklabels(top_xgb['feature'])
axes[1].set_xlabel('Importance (Gain)')
axes[1].set_title('Top 10 Features: XGBoost Gain Importance')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nNote: Permutation importance directly measures impact on model performance,")
print("while XGBoost's gain importance measures contribution to loss reduction.")
print("Both are useful, but permutation importance is in units of the metric.")


### SHAP Values (SHapley Additive exPlanations)

SHAP is a unified framework for interpreting predictions, based on Shapley values from cooperative game theory. It provides both local (per-prediction) and global (overall) explanations.


In [ ]:
# SHAP Values (if available)
if SHAP_AVAILABLE:
    print("Calculating SHAP values...")
    
    # Create SHAP explainer
    explainer = shap.TreeExplainer(model)
    
    # Calculate SHAP values for a sample (to save computation time)
    sample_size = min(100, len(X_test))
    X_sample = X_test.iloc[:sample_size]
    shap_values = explainer.shap_values(X_sample)
    
    # For binary classification, shap_values might be 2D array
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Use positive class SHAP values
    
    print(f"SHAP values calculated for {sample_size} samples")
    print(f"SHAP values shape: {shap_values.shape}")
    
    # Summary plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, feature_names=X.columns.tolist(), show=False)
    plt.title('SHAP Summary Plot (Top Features)')
    plt.tight_layout()
    plt.show()
    
    # Feature importance from SHAP (mean absolute SHAP value)
    shap_importance = pd.DataFrame({
        'feature': X.columns.tolist(),
        'importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Features by SHAP Importance (mean |SHAP|):")
    print(shap_importance.head(10).to_string(index=False))
    
    # Compare all three importance methods
    comparison_imp = pd.DataFrame({
        'feature': X.columns.tolist(),
        'xgb_gain': feature_importance.set_index('feature').loc[X.columns, 'importance'].values,
        'permutation': perm_df.set_index('feature').loc[X.columns, 'importance_mean'].values,
        'shap': shap_importance.set_index('feature').loc[X.columns, 'importance'].values
    })
    
    # Normalize for comparison
    for col in ['xgb_gain', 'permutation', 'shap']:
        comparison_imp[col] = comparison_imp[col] / comparison_imp[col].max()
    
    # Plot comparison for top 10 features
    top_features_comp = comparison_imp.nlargest(10, 'shap')
    
    x = np.arange(len(top_features_comp))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width, top_features_comp['xgb_gain'], width, label='XGBoost Gain', alpha=0.7)
    ax.bar(x, top_features_comp['permutation'], width, label='Permutation', alpha=0.7)
    ax.bar(x + width, top_features_comp['shap'], width, label='SHAP', alpha=0.7)
    
    ax.set_xlabel('Feature')
    ax.set_ylabel('Normalized Importance')
    ax.set_title('Feature Importance Comparison (Top 10)')
    ax.set_xticks(x)
    ax.set_xticklabels(top_features_comp['feature'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print("\nNote: SHAP values provide the most comprehensive interpretation:")
    print("  - Local: explains individual predictions")
    print("  - Global: mean |SHAP| gives overall feature importance")
    print("  - Directional: sign shows if feature increases or decreases prediction")
    
else:
    print("SHAP not available. Install with: pip install shap")
    print("\nSHAP provides:")
    print("  - Local explanations (per prediction)")
    print("  - Global feature importance")
    print("  - Directional effects (positive/negative contributions)")
    print("  - Interaction effects (advanced)")
    print("\nFor XGBoost, TreeExplainer is very efficient and recommended.")


## Deployment & Model Management

After developing a solid XGBoost model, you'll want to deploy it for production use. This section covers saving/loading models, pipeline integration, and serving strategies.


### Saving and Loading Models

XGBoost models can be saved in multiple formats for deployment:


In [ ]:
# Save model in JSON format (human-readable, recommended)
model_path_json = "outputs/results/xgb_model.json"
model.save_model(model_path_json)
print(f"Model saved to: {model_path_json}")

# Load model
model_loaded = XGBClassifier()
model_loaded.load_model(model_path_json)

# Verify loaded model works
y_pred_loaded = model_loaded.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)

print(f"\nModel Loading Verification:")
print(f"  Original model accuracy: {accuracy:.3f}")
print(f"  Loaded model accuracy: {accuracy_loaded:.3f}")
print(f"  Match: {np.allclose(y_pred, y_pred_loaded)}")

# Save using joblib (alternative, preserves sklearn wrapper)
import joblib
model_path_joblib = "outputs/results/xgb_model.joblib"
joblib.dump(model, model_path_joblib)
print(f"\nModel also saved to: {model_path_joblib} (joblib format)")

# Model file size
import os
json_size = os.path.getsize(model_path_json) / 1024  # KB
print(f"  JSON model size: {json_size:.2f} KB")

# Save model metadata
from src.utils.traceability import save_traceability_data
traceability_data = {
    'model_type': 'XGBClassifier',
    'n_estimators': model.n_estimators,
    'learning_rate': model.learning_rate,
    'max_depth': model.max_depth,
    'test_accuracy': float(accuracy),
    'feature_importance': feature_importance.head(10).to_dict('records'),
    'best_iteration': model_early.best_iteration if hasattr(model_early, 'best_iteration') else None
}
trace_path = save_traceability_data(traceability_data, "xgboost_trace")
print(f"\nTraceability data saved to: {trace_path}")


### Integration with Scikit-Learn Pipelines

XGBoost integrates seamlessly with sklearn pipelines:


In [ ]:
# Create pipeline with preprocessing and XGBoost
# Note: XGBoost doesn't require scaling, but this demonstrates pipeline usage
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Optional for XGBoost
    ('xgb', XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# Train pipeline
pipeline.fit(X_train, y_train)

# Evaluate
y_pred_pipeline = pipeline.predict(X_test)
acc_pipeline = accuracy_score(y_test, y_pred_pipeline)

print("Pipeline Results:")
print(f"  Test Accuracy: {acc_pipeline:.3f}")

# Cross-validate pipeline
cv_scores_pipeline = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"  CV Accuracy: {cv_scores_pipeline.mean():.3f} (+/- {cv_scores_pipeline.std():.3f})")

# Save entire pipeline
pipeline_path = "outputs/results/xgb_pipeline.joblib"
joblib.dump(pipeline, pipeline_path)
print(f"\nFull pipeline saved to: {pipeline_path}")

# Load and use pipeline
pipeline_loaded = joblib.load(pipeline_path)
y_pred_pipeline_loaded = pipeline_loaded.predict(X_test)
print(f"  Loaded pipeline accuracy: {accuracy_score(y_test, y_pred_pipeline_loaded):.3f}")

print("\nNote: Pipelines ensure consistent preprocessing in production.")
print("The scaler will be applied automatically to new data.")


### Performance Considerations for Deployment

When deploying XGBoost models, consider:


In [ ]:
# Benchmark prediction speed
import time

# Single prediction
start = time.time()
for _ in range(1000):
    _ = model.predict(X_test.iloc[0:1])
single_pred_time = (time.time() - start) / 1000

# Batch prediction
start = time.time()
_ = model.predict(X_test)
batch_pred_time = (time.time() - start) / len(X_test)

print("Prediction Performance:")
print(f"  Single prediction: {single_pred_time*1000:.3f} ms")
print(f"  Batch prediction (per sample): {batch_pred_time*1000:.3f} ms")
print(f"  Batch throughput: {len(X_test)/batch_pred_time:.0f} predictions/second")

# Model size
model_size_kb = os.path.getsize(model_path_json) / 1024
print(f"\nModel Characteristics:")
print(f"  Model size: {model_size_kb:.2f} KB")
print(f"  Number of trees: {model.n_estimators}")
print(f"  Max depth: {model.max_depth}")
print(f"  Number of features: {X.shape[1]}")

# Memory usage estimate
# Each tree node stores: feature index, threshold, left/right child, value
# Rough estimate: ~20 bytes per node
estimated_nodes = model.n_estimators * (2 ** (model.max_depth + 1))
estimated_memory_mb = (estimated_nodes * 20) / (1024 * 1024)
print(f"  Estimated memory (rough): {estimated_memory_mb:.2f} MB")

print("\nDeployment Tips:")
print("  - XGBoost predictions are very fast (milliseconds per instance)")
print("  - For high-throughput, batch predictions are more efficient")
print("  - Model size is typically small (KB to MB)")
print("  - For extreme low latency, consider model compression or Treelite")


## Advanced Topics: Custom Objectives, Custom Metrics, and Performance Optimization

For most users, XGBoost's built-in capabilities suffice. But if you need to extend XGBoost or squeeze out more performance, here are advanced topics:


### Custom Objective Function

XGBoost allows you to define your own optimization objective, as long as you can provide the first and second derivative of the loss function.


In [ ]:
# Example: Custom logistic objective (for demonstration - XGBoost already has this)
# This shows the form of a custom objective function

def custom_logistic_obj(preds, dtrain):
    """
    Custom objective function for binary logistic regression.
    Returns gradient and Hessian.
    
    Args:
        preds: Current predictions (logits, not probabilities)
        dtrain: DMatrix with training data
    
    Returns:
        grad: First derivative (gradient)
        hess: Second derivative (Hessian)
    """
    y_true = dtrain.get_label()
    
    # Convert logits to probabilities using sigmoid
    preds_prob = 1.0 / (1.0 + np.exp(-preds))
    
    # Gradient: derivative of logloss with respect to logit
    grad = preds_prob - y_true
    
    # Hessian: second derivative (p * (1-p))
    hess = preds_prob * (1 - preds_prob)
    
    return grad, hess

# Example usage with low-level API
print("Custom Objective Function Example:")
print("  This demonstrates how to create a custom objective.")
print("  For binary classification, XGBoost's built-in 'binary:logistic' is recommended.")

# Note: We won't actually train with this since XGBoost already has logistic
# But this shows the pattern for other custom objectives (e.g., focal loss, quantile loss)

# Example custom objective usage (commented out - for reference):
# params_custom = {'max_depth': 3, 'learning_rate': 0.1, 'random_state': 42}
# bst_custom = xgb.train(
#     params_custom,
#     dtrain,
#     num_boost_round=100,
#     obj=custom_logistic_obj,  # Custom objective
#     evals=[(dval, 'eval')]
# )

print("\nCustom objectives are useful for:")
print("  - Focal loss (for imbalanced data)")
print("  - Quantile regression (though XGBoost has this built-in)")
print("  - Custom ranking losses")
print("  - Domain-specific loss functions")


### Custom Evaluation Metric

If you want XGBoost to display a custom metric during training (and use it for early stopping), you can provide a custom eval function:


In [ ]:
# Example: Custom metric (Gini coefficient, which is 2*AUC - 1)
def gini_eval(preds, dtrain):
    """
    Custom evaluation metric: Gini coefficient.
    For binary classification, Gini = 2 * AUC - 1
    
    Args:
        preds: Current predictions (probabilities for binary classification)
        dtrain: DMatrix with training data
    
    Returns:
        Tuple of (metric_name, metric_value, is_higher_better)
    """
    y_true = dtrain.get_label()
    
    # Calculate AUC
    try:
        auc = roc_auc_score(y_true, preds)
        gini = 2 * auc - 1
        return 'gini', gini, True  # True means higher is better
    except:
        # Fallback if AUC calculation fails
        return 'gini', 0.0, True

# Example usage with low-level API
print("Custom Evaluation Metric Example:")
print("  Gini coefficient = 2 * AUC - 1")
print("  This can be used for monitoring during training.")

# Note: For sklearn API, you can't directly use custom eval in fit(),
# but you can monitor after training or use callbacks

# Example with low-level API (commented - for reference):
# params_metric = {
#     'objective': 'binary:logistic',
#     'max_depth': 3,
#     'learning_rate': 0.1,
#     'random_state': 42
# }
# bst_metric = xgb.train(
#     params_metric,
#     dtrain,
#     num_boost_round=100,
#     evals=[(dval, 'eval')],
#     feval=gini_eval,  # Custom evaluation metric
#     maximize=True,    # Gini should be maximized
#     early_stopping_rounds=10
# )

print("\nCustom metrics are useful for:")
print("  - Domain-specific metrics")
print("  - Business metrics (e.g., profit, cost-sensitive metrics)")
print("  - Metrics not built into XGBoost")


### Performance Optimizations

XGBoost offers several ways to optimize for speed and memory:


In [ ]:
# Compare different tree methods
print("Tree Method Comparison:")
print("  'exact': Exact greedy algorithm (default for small data)")
print("  'hist': Histogram-based (faster for large data)")
print("  'approx': Approximate algorithm")
print("  'gpu_hist': GPU-accelerated histogram (requires GPU)")

# Test histogram method (faster for larger datasets)
import time

# Exact method
start = time.time()
model_exact = XGBClassifier(
    n_estimators=50,
    tree_method='exact',
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
model_exact.fit(X_train, y_train)
time_exact = time.time() - start

# Histogram method
start = time.time()
model_hist = XGBClassifier(
    n_estimators=50,
    tree_method='hist',
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
model_hist.fit(X_train, y_train)
time_hist = time.time() - start

print(f"\nTraining Time Comparison (50 trees):")
print(f"  Exact method: {time_exact:.3f}s")
print(f"  Histogram method: {time_hist:.3f}s")
print(f"  Speedup: {time_exact/time_hist:.2f}x")

# Accuracy comparison
acc_exact = accuracy_score(y_test, model_exact.predict(X_test))
acc_hist = accuracy_score(y_test, model_hist.predict(X_test))
print(f"\nAccuracy Comparison:")
print(f"  Exact method: {acc_exact:.3f}")
print(f"  Histogram method: {acc_hist:.3f}")
print(f"  Difference: {abs(acc_exact - acc_hist):.4f}")

print("\nPerformance Tips:")
print("  1. Use tree_method='hist' for large datasets (often faster with minimal accuracy loss)")
print("  2. Use dtype=np.float32 instead of float64 to reduce memory")
print("  3. Reduce max_bin (e.g., 64 instead of 256) for histogram method to save memory")
print("  4. Use subsample and colsample_bytree < 1.0 to speed up training")
print("  5. For very large data, consider out-of-core computation with DMatrix from file")
print("  6. GPU training (tree_method='gpu_hist') can be much faster if GPU is available")


## Feature Importance

XGBoost provides multiple ways to measure feature importance. Let's explore them.


### Comprehensive Parameter Reference

**Booster Types:**
- `booster`: Type of booster ('gbtree', 'gblinear', 'dart'). Default is 'gbtree'

**Objective Functions:**
- `objective`: Loss function to optimize
  - Classification: 'binary:logistic', 'multi:softprob', 'multi:softmax'
  - Regression: 'reg:squarederror', 'reg:absoluteerror', 'reg:pseudohubererror'
  - Others: 'count:poisson', 'survival:cox', etc.

**Evaluation Metrics:**
- `eval_metric`: Metric to monitor during training
  - Classification: 'logloss', 'auc', 'error', 'merror' (multiclass error)
  - Regression: 'rmse', 'mae', 'rmsle' (root mean squared log error)

**Core Parameters:**
- `n_estimators` (num_boost_round): Number of boosting rounds
- `max_depth`: Maximum tree depth (default: 6)
- `learning_rate` (eta): Step size shrinkage (default: 0.3)
- `gamma` (min_split_loss): Minimum loss reduction for split (default: 0)
- `min_child_weight`: Minimum sum of instance weights in child (default: 1)

**Sampling Parameters:**
- `subsample`: Row sampling ratio (default: 1.0)
- `colsample_bytree`: Column sampling ratio per tree (default: 1.0)
- `colsample_bylevel`: Column sampling ratio per level (default: 1.0)
- `colsample_bynode`: Column sampling ratio per node (default: 1.0)

**Regularization:**
- `reg_alpha` (alpha): L1 regularization (default: 0)
- `reg_lambda` (lambda): L2 regularization (default: 1.0)

**Imbalanced Data:**
- `scale_pos_weight`: Scale gradient for positive class (default: 1.0)
- `max_delta_step`: Maximum delta step for logistic regression (default: 0)

**Tree Construction:**
- `tree_method`: Algorithm ('exact', 'approx', 'hist', 'gpu_hist')
- `max_bin`: Maximum number of bins for histogram (default: 256)
- `grow_policy`: Tree growth policy ('depthwise', 'lossguide')

**Other:**
- `random_state`: Random seed for reproducibility
- `n_jobs`: Number of parallel threads (default: -1, use all cores)
- `early_stopping_rounds`: Stop if no improvement for N rounds


## Implementation

Let's implement XGBoost for both classification and regression.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.models.ensemble import extract_feature_importance, plot_feature_importance
from src.utils.benchmarking import benchmark_model_training
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Classification Example: Breast Cancer
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='Target')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {cancer.target_names.tolist()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Train XGBoost Classifier
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)

print("\nXGBoost Classifier:")
print(f"Number of estimators: {model.n_estimators}")
print(f"Learning rate: {model.learning_rate}")
print(f"Max depth: {model.max_depth}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Early Stopping

Let's use early stopping to prevent overfitting.


In [ ]:
# XGBoost with early stopping
model_early = XGBClassifier(
    n_estimators=1000,  # Large number, will stop early
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=10
)

# Split training data for validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

model_early.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"Best iteration: {model_early.best_iteration}")
print(f"Best score: {model_early.best_score:.3f}")

# Make predictions
y_pred_early = model_early.predict(X_test)
accuracy_early = accuracy_score(y_test, y_pred_early)

print(f"\nTest Accuracy (with early stopping): {accuracy_early:.3f}")
print(f"Number of trees used: {model_early.best_iteration + 1}")


## Validation & Testing

Let's validate the model and compare with standard Gradient Boosting.


In [ ]:
# Validation 1: Compare with scikit-learn Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)

print("Comparison: XGBoost vs Gradient Boosting")
print(f"  XGBoost: {accuracy:.3f}")
print(f"  Gradient Boosting: {gb_accuracy:.3f}")
print(f"  Improvement: {accuracy - gb_accuracy:.3f}")

# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"\nCross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 3: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > 0.5, "Accuracy should be better than random!"
print("\n✓ Validation checks passed")


## Feature Importance

Let's extract and visualize feature importance.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize
plot_feature_importance(feature_importance, top_n=15, title="XGBoost Feature Importance")


## Regression Example

Let's apply XGBoost to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg, y_reg, test_size=0.2, random_state=42)

# Train XGBoost Regressor
xgb_reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='rmse'
)
xgb_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = xgb_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("XGBoost Regression:")
print(f"  Test RMSE: {rmse:.3f}")

# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.6)
plt.plot([y_reg_test.min(), y_reg_test.max()], 
         [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('XGBoost Regression: Predicted vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Real-World Application

Let's tune hyperparameters and compare regularization effects.


In [ ]:
# Test regularization parameters
reg_alpha_values = [0, 0.1, 0.5, 1.0]
reg_lambda_values = [0, 0.1, 0.5, 1.0]

print("Testing Regularization Parameters:")
for alpha in reg_alpha_values:
    for lam in reg_lambda_values:
        xgb = XGBClassifier(
            n_estimators=50,
            learning_rate=0.1,
            max_depth=3,
            reg_alpha=alpha,
            reg_lambda=lam,
            random_state=42,
            eval_metric='logloss'
        )
        xgb.fit(X_train, y_train)
        pred = xgb.predict(X_test)
        acc = accuracy_score(y_test, pred)
        print(f"  alpha={alpha}, lambda={lam}: Accuracy = {acc:.3f}")

# Hyperparameter tuning (reduced grid for speed)
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [0, 0.1]
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **XGBoost Basics**
   - Optimized gradient boosting algorithm
   - Includes L1 and L2 regularization
   - Parallel tree construction
   - Handles missing values automatically

2. **Key Optimizations**
   - **Regularization**: Prevents overfitting (alpha, lambda)
   - **Parallel Processing**: Faster training
   - **Tree Pruning**: Level-wise growth with backward pruning
   - **Approximate Split Finding**: Faster than exact search
   - **Sparsity Awareness**: Efficient with sparse data

3. **Key Hyperparameters**
   - **n_estimators**: Number of trees
   - **learning_rate**: Step size (lower = more trees needed)
   - **max_depth**: Tree complexity
   - **reg_alpha**: L1 regularization
   - **reg_lambda**: L2 regularization
   - **subsample**: Row sampling
   - **colsample_bytree**: Column sampling

4. **Best Practices**
   - Use early stopping to prevent overfitting
   - Start with default parameters
   - Tune learning_rate and n_estimators together
   - Use regularization for overfitting
   - Monitor validation performance

### When to Use XGBoost

✅ **Good for:**
- Highest accuracy requirements
- Large datasets
- Structured/tabular data
- Competition-level performance
- Both classification and regression
- When you have computational resources

❌ **Not ideal for:**
- Very small datasets (may overfit)
- When interpretability is crucial
- Real-time predictions (can be slow)
- Text/image data (use neural networks)
- When simplicity is preferred

### Next Steps

- Try **LightGBM** for faster training on large datasets
- Explore **CatBoost** for categorical features
- Use **SHAP values** for model interpretability
- Compare with **Random Forest** and **Gradient Boosting**
- Apply to **Kaggle competitions** for practice
